In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [14]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [15]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [16]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Nice—leftover chicken and rice are everything you need for quick, versatile meals. Here are some tasty options you can make, with quick descriptions and links to full recipes if you want to follow a step-by-step.

1) Chicken Fried Rice (quick, in one pan)
- Why it’s great: Uses both leftovers, comes together fast, and you can customize with veggies you have on hand.
- What you’ll typically need: leftover chicken, leftover rice, soy sauce, oil, eggs, and any veggies (peas, carrots, onions). Optional sesame oil, green onions.
- Quick picks:
  - Pioneer Woman’s Chicken Fried Rice
  - Allrecipes Easy Chicken Fried Rice
  - Easy Peasy Foodie: Leftover Chicken and Egg Fried Rice
  - Quick version: The Dinner Bite or Quick Weeknight Meals
- Links:
  - https://www.thepioneerwoman.com/food-cooking/recipes/a61241480/chicken-fried-rice-recipe/
  - https://www.allrecipes.com/easy-chicken-fried-rice-recipe-8644579
  - https://www.easypeasyfoodie.com/leftover-chicken-egg-fried-rice/
  - https://quic

In [17]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='83700a99-8130-4734-a6d0-32bbd2201937'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 460, 'prompt_tokens': 199, 'total_tokens': 659, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dg3qCaHk3nyMOPeSI1rvIejur3vvH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2fb5-e7b8-7691-abf1-01ee2df79f29-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes using leftover chicken and rice'}, 'id': 'call_IoXkN7ONcFFB7ybeVmEse4HR', 'type': 'tool_call'}, {'

In [ ]:
import base64
from pathlib import Path


def get_image_file(path: Path):
    img_bytes = path.read_bytes()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    return img_b64

img_b64
# Get the first (and only) uploaded file dict
uploaded_file = 
# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [7]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [19]:
from langchain.messages import HumanMessage
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Extract the list of ingredients you recognize from the attached image and divide them into categories. You will only return the list of ingredients and categories without additional text."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id